# Recommend counter-ions for a new drug (not in our data)
Give the SMILES of the **charged form** of the drug (protonated base = cation, deprotonated acid = anion) and, if you have it, a **measured** pKa (for a base: the pKa of its conjugate acid). The notebook ranks every counter-ion of the opposite charge with the final score equation (2D + 3D similarity, pKa, ESP) and writes `new_drug_result.json`, which you can load in the web app (box 'Your own drug').

Notes: ESP is not computed here (needs xTB), so that group stays neutral (about 4% of the score). Without a measured pKa the pKa group stays neutral. Drug pKa is never predicted.

In [ ]:
!pip -q install rdkit lightgbm xgboost scikit-learn pandas matplotlib
from google.colab import files
print('Upload code_and_data.zip from the submission folder')
up = files.upload()
!unzip -q -o code_and_data.zip -d work
%cd work/code_and_data
!ls

## Your drug

In [ ]:
SMILES = 'CC(C)Cc1ccc(cc1)C(C)C(=O)[O-]'   # example: ibuprofen anion. Replace with your drug (charged form)
NAME   = 'my drug'
PKA    = 4.5                               # measured pKa, or None if you do not have one
TOP    = 15

## Run

In [ ]:
import pandas as pd
from recommend_new_drug import recommend, to_app_json
R, meta = recommend(SMILES, name=NAME, pKa=PKA)
print(meta)
if meta['same_molecule_in_data']: print('Note: this molecule is already in our data as', meta['same_molecule_in_data'], '- all its forms were left out, so this is a fair new-drug answer.')
pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 60)
R.head(TOP)[['rank','ion_name','ion_smiles','score','pKa_label','sim2D','sim3D','similar_known_drugs_using_ion']]

## Save for the web app and download

In [ ]:
R.to_csv('new_drug_ranking.csv', index=False)
to_app_json(R, meta, 'new_drug_result.json')
files.download('new_drug_result.json'); files.download('new_drug_ranking.csv')

How to read the table: `score` = 100 x estimated probability that the ion is a reported counter-ion for this drug (the average candidate scores about 2.8). The four `c_` columns in the CSV show how much each feature group adds. `similar_known_drugs_using_ion` lists the most similar known drugs that already have this counter-ion published. ΔpKa labels are information only and never remove an ion.